# Fractal Breakout

## Contents

- [Configuration](#configuration)
  - [Setup](#setup)
  - [Automatic](#automatic)
  - [Manual](#manual)
  - [Final configuration](#final-configuration)
- [Fractal Breakout](#fractal-breakout)
  - [Backtesting](#fractal-breakout-backtesting)
  - [Grid search](#fractal-breakout-grid-search)
  - [Walk-forward analysis](#fractal-breakout-walk-forward)
  - [Monte Carlo simulations](#fractal-breakout-monte-carlo)
  - [Live signals](#fractal-breakout-live-signals)
- [Inverse Fractal Breakout](#inv-fractal-breakout)
  - [Backtesting](#inv-fractal-breakout-backtesting)
  - [Grid search](#inv-fractal-breakout-grid-search)
  - [Walk-forward analysis](#inv-fractal-breakout-walk-forward)
  - [Monte Carlo simulations](#inv-fractal-breakout-monte-carlo)
  - [Live signals](#inv-fractal-breakout-live-signals)

Fractal Breakout (Price Action) \
Detects S/R from N-bar fractal pivots — indicators.detect_swing_highs / detect_swing_lows (a strict left/right window), merged with indicators.merge_price_levels. \
This is the indicators-layer level source. It is not the dedicated engine.level_detector — that stateful horizontal-S/R detector (with invalidation tracking) powers the separate level_breakout strategy (see level_breakout.ipynb). \
Uses ATR(14) for the trailing stop. Best for scalping (1m-5m) and intraday (15m-1h).

How the Fractal-Breakout Algorithm Determines Entry/Exit:
- Detects swing highs (resistance) and lows (support) as N-bar fractal pivots with a configurable left/right window.
- Long Entry: Price closes above a significant resistance level + confirmation candle.
- Short Entry: Price closes below a significant support level + confirmation candle.
- Exit Long: Price closes below next support or ATR trailing stop hit.
- Exit Short: Price closes above next resistance or ATR trailing stop hit.
- Ideal for scalping/intraday when price respects liquidity levels.

## Configuration

### Setup

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import dataclasses

from engine.backtester import Backtester
from engine.data_configurator import ACTIVE, load_data, save_result, LIVE_DIR
from engine.strategy_configurator import params_for, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE
from engine.visualization import build_chart
from engine.evaluation import walk_forward, monte_carlo, grid_search
from engine.live import LiveEngine

import pandas as pd
import plotly.express as px

### Automatic

In [ ]:
# Automatic config: project-wide defaults defined by the three configurators.
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = params_for("fractal_breakout")   # engine/strategy_configurator.py (FractalParams — this notebook's family)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

### Manual


*_CONFIG = Automatic defaults, with any Manual overrides layered on top:
- Leave *_OVERRIDES empty → *_CONFIG is pure Automatic.
- Fill it → Automatic baseline + your Manual overrides.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic FractalParams().
# A foreign key raises TypeError here, not a silent no-op.
STRATEGY_OVERRIDES = {}      # e.g. {"left": 3, "right": 3}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

### Final configuration

In [ ]:
# Prepare the final inputs the rest of the notebook uses.
# Runs after overrides.
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

# Report exactly what data + which configs are in force downstream (manual or automatic).
_window = (f"{DATA_CONFIG.start} → {DATA_CONFIG.end or 'now'}"
           if DATA_CONFIG.is_range else f"last {DATA_CONFIG.num_candles}")
tc = TRADING_CONFIG
_exits = (", ".join(f"{k!r}: {v!r}" for k, v in STRATEGY_CONFIG.EXITS.items())
          if EXIT_POLICY is None else f"override → {EXIT_POLICY}")
print(f"Loaded {len(df):,} candles | {SYMBOL} {INTERVAL}m {DATA_CONFIG.category} | "
      f"{_window} | {df.index[0]:%Y-%m-%d %H:%M} → {df.index[-1]:%Y-%m-%d %H:%M} UTC")
print(f"Trade: initial_equity={tc.initial_equity}, position_size_bps={tc.position_size_bps}, "
      f"leverage={tc.leverage}, sizing_mode={tc.sizing_mode.value!r}, "
      f"risk_per_trade_bps={tc.risk_per_trade_bps}, direction={tc.direction.value!r}")
print("Strategy Parameters: "
      + ", ".join(f"{k}={v}" for k, v in dataclasses.asdict(STRATEGY_CONFIG).items()))
print(f"Strategy exits: {_exits}")

<a id="fractal-breakout"></a>
## Fractal Breakout

It's a direction flip, not different entry logic:
- fractal_breakout has one entry signal: breakout (close crosses a fractal-pivot-derived S/R level).
- fractal_breakout and fractal_breakout_inv are two separate classes — ride the breakout vs fade it.
- The _inv is the same signal traded in the opposite direction.

<a id="fractal-breakout-backtesting"></a>
### Backtesting

In [ ]:
# Import Fractal Breakout strategy
from engine.strategies import FractalBreakoutStrategy
STRATEGY = FractalBreakoutStrategy

In [ ]:
# Backtest Fractal Breakout strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="fractal_breakout")

In [ ]:
# Fractal Breakout strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="fractal-breakout-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"left": [3, 5, 8], "right": [3, 5, 8]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per left × right cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"right", "left"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="left", columns="right", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="right", y="left", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs left × right swept in the grid above — nothing to plot.")

<a id="fractal-breakout-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is an in-sample window swept for the best params.
# TEST_BARS is an out-of-sample window the winner is then tested on.
# OBJECTIVE  can be any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
# MIN_TRADES lets ignore in-sample combos with fewer trades (noise, not signal)

GRID = {"left": [3, 5, 8], "right": [3, 5, 8]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits) over the strategy's base-config chart.
# Each fold re-optimises on its train window; see wf.folds_frame() for the per-fold params.
build_chart(strategy.prepare(df), trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | {SYMBOL} {INTERVAL}m").show()

<a id="fractal-breakout-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="fractal-breakout-live-signals"></a>
### Live signals

From the CLI — a terminal running the same loop (it prints a file link to the auto-refreshing chart):

    python -m engine --strategy fractal_breakout --mode live --interval 15 --poll 30

From a notebook cell — below (run() prints a clickable chart link, then blocks):

In [ ]:
# Live mode runs the same strategy / config / costs as the backtest above.
# It generates signals (tells you when to enter / exit). It does not place orders.
# run() prints a clickable link to the chart, which auto-refreshes every poll_seconds.
# engine.run() blocks the execution of the rest of the notebook until stopped.

live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

<a id="inv-fractal-breakout"></a>
## Inverse Fractal Breakout

<a id="inv-fractal-breakout-backtesting"></a>
### Backtesting

In [ ]:
# Import Inverse Fractal Breakout strategy
from engine.strategies import InverseFractalBreakoutStrategy
STRATEGY = InverseFractalBreakoutStrategy

In [ ]:
# Backtest Inverse Fractal Breakout strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Per-trade dollar P&L
trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "avg_duration_min": round(t.duration.total_seconds() / 60, 1) if t.duration else None,
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])
display(trades_pnl.head())

# Save metrics (JSON) + per-trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG, name="fractal_breakout_inv")

In [ ]:
# Inverse Fractal Breakout strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="inv-fractal-breakout-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.

gs = grid_search(
    STRATEGY,
    strategy_grid={"left": [3, 5, 8], "right": [3, 5, 8]},
    trade_grid={"leverage": [1.0, 2.0]},
    exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per left × right cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.

HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"right", "left"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="left", columns="right", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="right", y="left", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs left × right swept in the grid above — nothing to plot.")

<a id="inv-fractal-breakout-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is an in-sample window swept for the best params.
# TEST_BARS is an out-of-sample window the winner is then tested on.
# OBJECTIVE  can be any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
# MIN_TRADES lets ignore in-sample combos with fewer trades (noise, not signal)

GRID = {"left": [3, 5, 8], "right": [3, 5, 8]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample (is) and their out-of-sample (oos) performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold (top) + cross-fold stability summary (bottom).
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window.
# nunique==1 => the optimiser locked the same value every fold; wide min..max / large std => jumpy.
display(wf.param_stability())
wf.param_stability_summary()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.

eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
(px.line(eq, labels={"value": "equity", "index": ""}, color_discrete_sequence=["steelblue"],
         title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m")
 .update_layout(showlegend=False)
 .update_traces(hovertemplate="%{x}<br>%{y:,.2f}<extra></extra>",
                hoverlabel=dict(bgcolor="steelblue", font_color="white"))
 .show())

In [ ]:
# Walk-forward OOS trades (entries/exits) over the strategy's base-config chart.
# Each fold re-optimises on its train window; see wf.folds_frame() for the per-fold params.
build_chart(strategy.prepare(df), trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | {SYMBOL} {INTERVAL}m").show()

<a id="inv-fractal-breakout-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.

px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"}, color_discrete_sequence=["steelblue"],
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()

# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"}, color_discrete_sequence=["steelblue"],
             title="OOS max-drawdown distribution").show()

<a id="inv-fractal-breakout-live-signals"></a>
### Live signals

From the CLI — a terminal running the same loop (it prints a file link to the auto-refreshing chart):

    python -m engine --strategy fractal_breakout_inv --mode live --interval 15 --poll 30

From a notebook cell — below (run() prints a clickable chart link, then blocks):

In [ ]:
# Live mode runs the same strategy / config / costs as the backtest above.
# It generates signals (tells you when to enter / exit). It does not place orders.
# run() prints a clickable link to the chart, which auto-refreshes every poll_seconds.
# engine.run() blocks the execution of the rest of the notebook until stopped.

live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button